# <center>Tarea 8 — Aprendizaje no supervisado</center>

**Instrucciones:**

1. Nombra tu archivo `tarea_8.ipynb` (5 puntos).
2. Escribe tu nombre completo y, después, esta declaración:
   *Este trabajo es mío y sigue la integridad académica del Tec de
   Monterrey. La transcripción de IA que entrego corresponde
   íntegramente a este trabajo.*
3. Anota los nombres de los compañeros con quienes discutiste la
   tarea.
4. Uso de IA: permitido. Adjunta la transcripción completa de tus
   conversaciones (archivo aparte o celda al final). Sin
   transcripción, la tarea no se califica. Si no usaste IA,
   decláralo.
5. Incluye la sección final de **Verificación**.
6. Descarga tu `.ipynb` y súbelo a Canvas junto con la transcripción.

---

**Puntos por sección**

| Sección | Puntos |
|---|---|
| Compresión de imágenes (Vector Quantization) | 10 |
| Clustering para desarrollo económico | 20 |
| ¿Son reales estos grupos? | 15 |
| Agrupando países con K-Medoids | 20 |
| Agrupando series de tiempo | 20 |
| Verificación | 10 |
| Nombre del archivo, declaración y compañeros | 5 |
| **Total** | **100** |

---

Esta tarea es un laboratorio guiado: varias celdas están a medias y las completas tú. Hay dos
tipos de celda que **no debes modificar** (están marcadas); el resto es tuyo.

Una advertencia que vale para toda la tarea: los algoritmos de agrupamiento **siempre** devuelven
grupos. Si le pides cinco clusters a $k$-medias, te va a dar cinco clusters, tengan o no
significado — incluso si le das puro ruido. A diferencia del aprendizaje supervisado, aquí no hay
una respuesta correcta contra la cual comparar. Por eso, en esta tarea, justificar que los grupos
significan algo es tan importante como calcularlos.

Cualquiera de tus tareas entregadas puede tocarte en una **defensa
oral** (10 minutos, sin IA, con tu notebook abierto). Antes de entregar,
pregúntate: *¿podría explicar y modificar cada celda de esto?* Si una
celda no la entiendes, pídele al agente que te la explique — para eso sí
es buenísimo.

In [ ]:
## Escribe aquí tu nombre completo, la declaración requerida y con quién trabajaste.

In [ ]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

# Compresión de imágenes (Vector Quantization) (10 pts)

La cuantificación vectorial es una técnica de compresión de datos en la que todas las observaciones se sustituyen por su prototipo. Esto significa que en lugar de almacenar la matriz de datos  de tamaño $N\times D$, se almacena la matriz de prototipos de tamaño $K\times D$, y el vector código de asignaciones de clusters $C$. Aunque no es necesario obtener los prototipos mediante el algoritmo K-means, es una opción muy utilizada.
Como ejemplo, podemos considerar su aplicación a la compresión de imágenes. Para las imágenes en color, en las que cada píxel es un vector tridimensional de números enteros con valores entre 0 y 255, se puede agrupar por píxel tiene más sentido. En este caso, el cetroide de cada grupo representa un color que representa a un número grande de píxeles. El color de cada píxel se sustituye por el de su centriode.

A continuación de demuestra como hacer esto y el resultado en la imágen para diferentes valores de $k$.

Sube una foto o imagen propia y explora el resultado.
Reultilza el código proporcionado.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from PIL import Image

In [ ]:
# Cambia esta imagen por otra.
# Si tu imagen es muy grande, el agrupamiento podría demorar mucho. Contempla bajar su resolución.
img = np.asarray(Image.open('../data/flower.jpg'))

plt.imshow(img);

In [ ]:
# Explora el efecto de k. Reutiliza este código con tu propia imagen.
# Si tu imagen es muy grande, el agrupamiento podría demorar mucho: baja su resolución
# con algo como img = img[::2, ::2] antes de correr esta celda.

ks = [2, 3, 5, 10]

fig, axes = plt.subplots(1, len(ks) + 1, figsize=(4 * (len(ks) + 1), 4))

axes[0].imshow(img)
axes[0].set_title("Original")
axes[0].set_axis_off()

for ax, k in zip(axes[1:], ks):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(img.reshape(-1, 3))
    centros = km.cluster_centers_
    vectorized = labels.reshape(img.shape[0:2])
    ax.imshow(centros[vectorized] / 255)
    ax.set_title(f"k = {k}")
    ax.set_axis_off()

plt.tight_layout();

Contesta en la celda de abajo:

1. ¿A partir de qué valor de $k$ dejas de notar diferencia con el original en *tu* imagen? ¿De qué
   depende — del número de colores distintos, de qué tan grandes son las regiones de color plano,
   de otra cosa?
2. **Cuenta el ahorro.** Guardar la imagen original cuesta $N \times 3$ bytes (un byte por canal
   por píxel). Guardar la versión cuantizada cuesta $N$ asignaciones de cluster más $k \times 3$
   bytes de la paleta. Calcula la razón de compresión para el $k$ que elegiste y repórtala como un
   número. ¿Qué se perdió a cambio?

In [ ]:
# Tu cálculo de la razón de compresión aquí...

**Tus respuestas aquí** (edita esta celda de markdown)

# Clustering para desarrollo económico (20 pts)
Ejemplo tomado de *Data Science for Public Policy, Chen-Rubin-Cornwall, 2021*.

Las corporaciones de desarrollo económico y las cámaras de comercio apoyan a las comunidades locales atrayendo empleo e inversión. Dada la creciente necesidad de más puestos de trabajo en todo el país, las iniciativas de desarrollo económico son asuntos feroces, que a veces enfrentan a una comunidad contra otra en guerras de ofertas por los beneficios fiscales. 
Más allá del espectáculo de la guerra de ofertas, hay otras cuestiones que influyen en estas decisiones de ubicación.  Por un lado, la posible región anfitriona de los nuevos puestos de trabajo debe tener las condiciones económicas adecuadas para sostener y fomentar la nueva oportunidad. Supongamos que un ejecutivo tecnológico de Santa Clara o Alameda, en el Área de la Bahía de California, quisiera encontrar otro condado con condiciones socioeconómicas similares. La misma pregunta podría hacerse a la inversa para los promotores económicos: *¿cuáles son otras zonas que compiten directamente?*.

Un análisis podría considerar en primer lugar qué características observables de una ciudad o condado son puntos de venta para posibles empresas. ¿Es el tamaño de la población activa? ¿Es el tamaño relativo de la industria objetivo? En cualquiera de estos casos, los datos económicos disponibles públicamente pueden agruparse mediante la técnica $k$-means.

Para simplificar, definimos las industrias tecnológicas en línea utilizando los códigos SCIAN 5182, 5112, 5179, 5415, 5417 y 454111, aunque reconocemos que esto puede excluir subindustrias que están creciendo rápidamente en importancia en la tecnología.

## 1. Importa el conjunto de datos a nivel de condado que contiene información de más de 3.100 condados estadounidenses.

Los datos se construyen a partir de diversas bases de datos de la Oficina del Censo de EE.UU., en particular la Encuesta sobre la Comunidad Estadounidense, County Business Patterns y Small Area Income & Poverty Estimates.

- `fips`: Código del Sistema Federal de Procesamiento de la Información que asigna un identificador único a cada condado.
- `state` y `name`: Las abreviaturas del estado de EE.UU. y el nombre del condado.
- `all.emp`: empleo total.
- `pct.tech`: porcentaje de la población empleada en la industria tecnológica.
- `est`: porcentaje de establecimientos de empresas de esa industria.
- `pov`: tasa de pobreza.
- `inc`: renta media de los hogares.
- `ba`: porcentaje de personas con estudios universitarios.

In [ ]:
# Importa el dataset aquí, llama al dataframe cty
cty = pd.read_csv('../data/county_compare.csv')

Antes de aplicar $k$-medias, los datos deben estar centrados en la media y estandarizados para que ninguna entrada tenga una influencia desproporcionada en la agrupación (el supuesto de pesos iguales). 

Escala los campos numéricos y asigna el resultado a la variable `inputs`. Pudes usar la funcion `scale` de Scikit-Learn o escalar a mano.

In [ ]:
inputs = # COMPLETA

## 2. Agrupa usando K-Means y determina el valor de k

Usa valores de k entre 2 y 30. Genera un gráfica con la inercia y otro con el valor medio del coeficiente de silueta. Usa las gráficas para determinar el valor de k, almacena el valor óptimo de k en la variable `k_opt` y el modelo óptimo en `km_opt`.

In [ ]:
# ESCRIBE TU CODIGO AQUI

In [ ]:
# Sustituye tus respuestas aquí
k_opt = 
km_opt = 

¿Cúantos elementes contiene cada cluster? Muestra una pequeña tabla con la información.

In [ ]:
# Crea la tabla aquí


¿Tienen sentido estas agrupaciones? Supongamos que uno quiere crear una lista Buzzfeed de los condados más educados de EE.UU.. Es tan fácil como tomar la proporción de personas con un título universitario en cada condado y, a continuación, ordenar de mayor a menor. ¿Y si la lista se ampliara a los más educados y poblados? ¿O si además se añadieran los ingresos? La agrupación puede tratarse como una forma de agrupar observaciones con valores similares, lo que produce un ranking.

In [ ]:
# No modifiques este código, úsalo para explorar tus grupos.
# (Ojo: los ejes están en escala logarítmica. Antes de interpretar la figura, ve el chequeo 1
#  de la sección de Verificación.)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(np.log(cty.ba), np.log(cty['pct.tech']), c=km_opt.labels_, cmap='Accent',
                alpha=0.5, s=2 + 100*cty['all.emp']/cty['all.emp'].max())
axes[0].set_xlabel("log(% con licenciatura)")
axes[0].set_ylabel("log(% empleo en tecnología)")
axes[0].set_title("Empleo tecnológico y educación")

axes[1].scatter(np.log(cty.inc), np.log(cty['pct.tech']), c=km_opt.labels_, cmap='Accent',
                alpha=0.5, s=2 + 100*cty['all.emp']/cty['all.emp'].max())
axes[1].set_xlabel("log(ingreso mediano)")
axes[1].set_ylabel("log(% empleo en tecnología)")
axes[1].set_title("Empleo tecnológico e ingreso")

plt.tight_layout();

Los diagramas de dispersión sugieren que los centros de empleo más grandes tienden a tener también una mayor concentración de empleo tecnológico y mayores ingresos, lo que no es sorprendente dada la tendencia a la urbanización. Estas zonas ricas en recursos tienden a burbujear hacia arriba en todas las medidas. Es cierto que estos resultados no son causales, pero sí sugestivos.  En efecto, el algoritmo $k$-means ofrece una estrategia eficaz para separar las comunidades con más recursos de las que tienen menos.

¿Cómo deben utilizarse estos resultados? Depende del público. Una autoridad de desarrollo económico no necesita competir con todos los condados competidores, sino que debe centrarse en los comparables. Al agrupar todos los condados en función de sus características observables, los promotores económicos pueden situar su condado en un conjunto de condados similares. Si puede articular qué otros condados son competidores en su segmento de mercado, también podrá elaborar ofertas para las posibles empresas que se distingan del resto.

Una empresa tecnológica que busque el emplazamiento de su futura sede puede no querer pagar por estar en las ciudades más caras, pero sí desearía tener acceso a una mano de obra tecnológica cualificada y de tamaño razonable. $k$-means condensa los datos de competitividad en una simple lista de dónde se concentra la tecnología. El más pequeño de los clusters se compone de zonas icónicas de alta tecnología como San Francisco y Santa Clara en California, así como grandes ciudades como Nueva York (New York City) y Seattle (King County, WA). El mismo cluster encuentra alternativas menos costosas y menos densamente pobladas como Durham (NC) y Arlington (VA), ambas centros crecientes de excelencia técnica. En esencia, la agrupación puede identificar alternativas inteligentes para informar el proceso de selección. 

A pesar de estos prometedores casos de uso, hay que tener en cuenta sus limitaciones. Desde un punto de vista ético, el análisis de conglomerados basado en los resultados anteriores de las regiones podría ser una profecía autocumplida. Las zonas con buenos recursos seguirán atrayendo más atención y recursos, ignorando las zonas prometedoras. Además, la agrupación en función de datos pasados no proporciona ninguna indicación sobre el rendimiento futuro de las comarcas. En el caso de $k$-means, en particular, los resultados son "planos": las etiquetas de los grupos son unidimensionales e ignoran el rico contexto que muestra cómo algunas observaciones se parecen más a unas que a otras. Otros métodos podrían ser más útiles para captar ese contexto, como la agrupación jerárquica.

# ¿Son reales estos grupos? (15 puntos)

El texto de arriba es persuasivo, pero está escrito como si los cinco (o los que hayan sido)
grupos existieran allá afuera y el algoritmo los hubiera *encontrado*. No es así: tú elegiste
cuántos grupos habría, cuáles variables entraban y cómo se escalaban. Cambia cualquiera de esas
tres decisiones y cambian los grupos.

Esta sección es el contrapeso. Antes de que alguien tome una decisión de política pública con tus
clusters, hay que saber qué tan sólidos son.

### a. Justifica el número de grupos (5 pts)

1. Reporta el **valor numérico** del coeficiente de silueta medio de `km_opt`, no solo la gráfica.
   Reporta también el de $k_{opt}-1$ y $k_{opt}+1$. Si los tres son casi iguales, tu elección de
   $k$ es una preferencia, no un hallazgo — y así hay que decirlo.

2. Interpreta la magnitud. Un coeficiente de silueta cercano a 1 indica grupos compactos y bien
   separados; alrededor de 0 indica que los puntos están casi tan cerca de su grupo como del
   vecino. ¿En qué parte de ese rango cae el tuyo? ¿Qué te dice sobre si los condados forman
   verdaderos racimos o son una nube continua que estás cortando en rebanadas?

3. **La prueba del ruido.** Construye una copia de `inputs` en la que permutes *independientemente*
   cada columna (con `np.random.default_rng(42).permutation` columna por columna). Eso destruye
   toda relación entre variables pero conserva la distribución de cada una por separado — es decir,
   destruye la estructura de grupos si la había. Corre $k$-medias con tu misma $k$ sobre esos datos
   permutados y reporta su coeficiente de silueta.

   Compara los dos números. Si la silueta de tus datos reales no es claramente mayor que la del
   ruido, tus grupos no están capturando estructura: están capturando la forma de la nube.

In [ ]:
# a. Silueta de k_opt-1, k_opt, k_opt+1, y la prueba del ruido.

**Tus respuestas al inciso a aquí** (edita esta celda de markdown)

### b. Revisa el escalado (4 pts)

1. **Imprime la lista exacta de columnas que entraron en `inputs`.** Hazlo aunque estés seguro.
   Si construiste `inputs` con algo como `cty.select_dtypes(include=np.number)`, revisa qué se
   coló: `fips` es un número, pero es el identificador del condado, y sus primeros dígitos son el
   código del estado. Un clustering que incluya `fips` está agrupando parcialmente por geografía
   alfabética. ¿Está en tus `inputs`? ¿Debería?

2. **Muestra qué pasa sin escalar.** Corre $k$-medias con la misma $k$ sobre las columnas
   numéricas *sin* estandarizar, y compara los tamaños de los grupos resultantes contra los de
   `km_opt`. Explica el resultado: ¿cuál es el rango de `inc` (decenas de miles) contra el de
   `pov` (unidades)? ¿Cuál variable termina decidiendo prácticamente sola la agrupación cuando no
   escalas, y por qué?

In [ ]:
# b. Columnas de inputs, y comparación escalado vs. sin escalar.

**Tus respuestas al inciso b aquí** (edita esta celda de markdown)

### c. Corrige al agente (3 pts)

Una analista le pidió a una IA que redactara el resumen ejecutivo de este análisis para la
autoridad de desarrollo económico. Esto fue lo que devolvió:

> El análisis de $k$-medias identificó cinco tipos de condados económicamente distintos que
> existen en la economía estadounidense. El Cluster 3 —condados con alta concentración
> tecnológica, ingresos altos y baja pobreza— demuestra que invertir en la industria tecnológica
> reduce la pobreza: los condados de este grupo tienen tasas de pobreza 8 puntos porcentuales por
> debajo del promedio nacional. Recomendamos que los condados del Cluster 1 adopten políticas
> agresivas de atracción de empresas tecnológicas para migrar al Cluster 3 en los próximos cinco
> años. El modelo alcanzó una precisión del 87%, por lo que las asignaciones de condados a
> clusters son confiables.

Suena profesional y está completamente mal. **Identifica al menos tres afirmaciones injustificadas
y explica, en una o dos líneas cada una, por qué el análisis que hiciste no las sostiene.**
Después reescribe el párrafo para que sea defendible.

Pistas de dónde buscar, sin decirte cuáles son: piensa en qué significa "existen"; en qué se
necesita para afirmar "reduce"; en de dónde salió el número 8 y con qué se está comparando; en si
estos datos tienen dimensión temporal; y en qué querría decir "precisión" cuando no hay etiquetas
verdaderas contra las cuales comparar.

**Tu corrección aquí** (edita esta celda de markdown)

**Afirmación injustificada 1:** — por qué:

**Afirmación injustificada 2:** — por qué:

**Afirmación injustificada 3:** — por qué:

**Párrafo reescrito:**

### d. Memorándum (3 pts)

Escribe **cinco renglones** dirigidos a la directora de desarrollo económico de un condado, que no
sabe qué es $k$-medias y tiene que decidir si usa esto. Los cinco renglones deben incluir, sin
tecnicismos:

- qué es lo que sí puedes afirmar con estos grupos,
- qué es lo que **no** puedes afirmar con ellos,
- y una condición bajo la cual no deberían usarse.

Cinco renglones es un límite, no una sugerencia. La dificultad del ejercicio está en la brevedad:
si necesitas diez, todavía no tienes clara la conclusión.

**Tu memorándum aquí** (máximo cinco renglones)

1.
2.
3.
4.
5.

# Agrupando países con K-Medoids (20 pts)

Este ejemplo, tomado de Kaufman y Rousseeuw (1990), proviene de un estudio en el que se pidió a
estudiantes de ciencias políticas que proporcionaran medidas de disimilitud por pares para 12
países: Bélgica, Brasil, Chile, Cuba, Egipto, Francia, India, Israel, Estados Unidos, Unión de
Repúblicas Socialistas Soviéticas, Yugoslavia y Zaire. Los puntajes promedio de disimilitud se dan
en la tabla que vas a importar debajo.
Aplica agrupamiento de k-medoids a estas diferencias.
Ten en cuenta que la agrupación de K-means no se pudo aplicar porque sólo tenemos distancias en
lugar de observaciones sin procesar.

In [ ]:
#!pip install kmedoids
import kmedoids

## 1. Importa el data set

In [ ]:
# No necesitas cambiar este código
countries = pd.read_csv(
    'https://github.com/UPY-UL/U3-M1-L1-Prototype_clustering/raw/master/Data/countries.data',
    sep=r'\s+',
    names=[
        'BEL', 'BRA', 'CHI', 'CUB', 'EGY', 'FRA',
        'IND', 'ISR', 'USA', 'USS', 'YUG', 'ZAI'
    ]
)
countries.index = [
    'BEL', 'BRA', 'CHI', 'CUB', 'EGY', 'FRA',
    'IND', 'ISR', 'USA', 'USS', 'YUG', 'ZAI'
]
countries

## 2. Aplica k-medoids clustering y determina el valor óptimo de k. Recuerda que estás trabajando con una matrix de distancias, explora la documentación de la función para averiguar como utilizar esta matriz directamente en lugar de calcular las distancias entre observaciones. *Nota: no uses el silhoutte plot, la clase KMedoids no es compatible con la función vista en clase.*

In [ ]:
# Almacena el valor optimo de k y el modelo óptimo en estas variables
k_opt = 
km_opt = 

## 3. Visualiza los clusters utilizando un MDS plot. Aquí no tienes que hacer nada, sólo interpretar el plot para el valor de k correcto.

La figura muestra un diagrama bidimensional de escala multidimensional (MDS), con las asignaciones de grupos indicadas por colores.
El escalado multidimensional (MDS) busca una representación de baja dimensión de los datos en la que las distancias respeten bien las distancias en el espacio original de alta dimensión.
Es particularmente útil cuando solo contamos con la matriz de dsitancia y queremos encontrar una representación aproximada de las observaciones originales.

In [ ]:
from sklearn import __version__ as sklearn_version
from sklearn.manifold import MDS

# El nombre del argumento para pasar una matriz de distancias ya calculada cambió en
# scikit-learn 1.8: `dissimilarity` quedó obsoleto en favor de `metric`.
if tuple(int(p) for p in sklearn_version.split(".")[:2]) >= (1, 8):
    compat = {"metric": "precomputed", "init": "classical_mds", "n_init": 1}
else:
    compat = {"dissimilarity": "precomputed", "n_init": 4}

embedding = MDS(n_components=2, normalized_stress='auto', random_state=42, **compat)
X_t = embedding.fit_transform(countries.values)

plt.scatter(X_t[:, 0], X_t[:, 1], c=km_opt.labels_)
for (x, y), country in zip(X_t, countries.columns):
    plt.annotate(country, (x, y));

# Agrupando series de tiempo (20 pts)

¿Cómo va la economía? ¿Dónde debemos concentrar nuestros esfuerzos? 

Aunque se trata de preguntas vagas, los responsables políticos se enfrentan a ellas con frecuencia.
Las preguntas implican la necesidad de comparar la economía con sus resultados históricos en una serie temporal, un problema bastante sencillo.
Pero cuando se amplía a la dimensión *dónde*, de repente debemos incorporar un componente geográfico que aumentará fácilmente la complejidad del problema.
Un análisis a nivel estatal amplía el número de series temporales a 50, mientras que el análisis a nivel de condado se dispara a 3.100 series temporales.
El reto de pasar de una sola serie temporal a miles es la enorme cantidad de información que hay que resumir de forma concisa, informativa y comprensible.
Pero tal vez cada condado no sea un copo de nieve.
Quizá cada condado pueda considerarse parte de un grupo económico.
Cada condado puede compararse con todos los demás utilizando las cualidades de sus series temporales (por ejemplo, ciclos estacionales, tendencias al alza o a la baja e irregularidades), y entonces los condados que se mueven de forma similar pueden formar parte del mismo grupo económico, es decir, identificar agrupaciones.
Estos clusters reducen la complejidad de los datos, de modo que sólo unos pocos perfiles distintos pueden constituir la base de un análisis económico conciso.

Aplicaras la agrupación jerárquica a los datos de series temporales para identificar y articular grupos de comportamiento.
Es una idea que puede aplicarse ampliamente a los datos económicos.
La agrupación puede ayudar a los responsables políticos a comprender que no existe un único tipo de crecimiento económico: es heterogéneo y algunas regiones pueden ser más resistentes que otras.
El enfoque puede aplicarse también a la ciberseguridad, para identificar tipos comunes de tráfico web y detectar actividades inusuales.
Prácticamente cualquier conjunto de series temporales puede beneficiarse de la agrupación. 

Usaremos el Censo Trimestral de Empleo y Salarios (QCEW), un conjunto de datos trimestrales sobre empleo y salarios que representa más del 95% de los puestos de trabajo de Estados Unidos.
Recopilados por la Oficina de Estadísticas Laborales de EE.UU., los datos son una de las principales fuentes de datos que cuentan la historia económica de EE.UU., mostrando todos los niveles de actividad económica desde el nivel de condado hasta la línea superior nacional.
La BLS no considera el QCEW como una serie temporal, pero contiene información útil si se trata como una serie temporal.
Aunque se publican aproximadamente 3.200 condados en el QCEW, ilustramos la agrupación en un subconjunto de datos, a saber, el empleo trimestral medio de los 58 condados de California.
Para facilitar el análisis, se han preprocesado los datos.
En primer lugar, los datos agregan los registros mensuales en registros trimestrales medios.
En segundo lugar, los datos también se han ajustado estacionalmente (SA), lo que significa que los ciclos normales de un año a otro se han extraído de los datos dejando sólo la tendencia y el ruido.
El conjunto de datos contiene 100 observaciones trimestrales desde el primer trimestre de 1992 hasta el cuarto trimestre de 2016.
Las series temporales se proporcionan en formato "wide", almacenando el índice de fecha y hora en las dos primeras columnas y las 58 columnas restantes contienen las series temporales de empleo a nivel de condado. 

## 1. Importa los datos

In [ ]:
# Importa los datos aquí
cali = pd.read_csv('../data/qcew_cali_sa.csv')

## 2. Crea la matriz de distancias

El proceso de agrupación jerárquica comienza con la construcción de una matriz de distancias que relaciona todos los puntos entre sí.
Las series temporales pueden asociarse entre sí en términos de correlación de Pearson, es decir, la frecuencia con la que un par de series temporales se mueven juntas. 
Transforma las 58 series temporales del marco de datos `cali` en una matriz de correlaciones de Pearson de $58 \times 58$.
Asigna el resultado a la variable `rho`. 

In [ ]:
rho = 

El algoritmo requiere una matriz de distancias a la que puedan aplicarse métodos de vinculación, y la forma actual de la matriz "rho" no se ajusta a ella. El conjunto de correlaciones de cada condado puede considerarse como coordenadas que indican la ubicación de una serie con respecto a otra, pero no como una distancia. Las correlaciones deben racionalizarse como una medida de distancia absoluta, lo que puede lograrse interpretando cada fila (o columna) de la matriz rho como un vector de coordenadas y calculando la matriz de distancias.
Calcula la matriz de distancias y asignala a la variable `d`.

In [ ]:
d = 

## 3. Agrupa

Usa agrupamiento jerárquico con el método de Ward para y grafica el dendrograma con el ordenamiento óptimo y etiquetas correctas.
Usa la seríe de gráficos al final de esta sección para encontrar un valor de $k$ adecuado. Justifica tu elección.

In [ ]:
# En esta celda puedes construir el árbol y graficar el dendrograma.


Explora diferentes particiones cambiando el valor de k, y creando un vector de etiquetas. Usa la función `fcluster` de scipy. Considera valores de menores o igual a 8.

In [ ]:
k = 
labels = 

In [ ]:
# No cambies este código, usa las gráficas para identificar un valor de k apropiado.
unique_l, c_sizes = np.unique(labels, return_counts=True)
max_size = c_sizes.max()
colors = plt.cm.Set2(np.linspace(0, 1, 8))

fig, axes = plt.subplots(k, max_size, figsize=(50, 10))

for ax in axes.ravel():
    ax.set_axis_off()

for l, n in zip(unique_l, c_sizes):
    cali_l = cali.iloc[:, 2:].loc[:, labels==l]
    for nn in range(n):
        axes[l-1, nn].set_title(cali_l.columns[nn])
        axes[l-1, nn].plot(cali_l.iloc[:, nn], color=colors[l-1])

**Justifica tu elección de k en este celda.**

*Texto aquí*

*¿Qué nos dicen estos gráficos?*
Estos clusters ilustran grupos económicos. Cada cluster tiene una especie de ritmo económico que distingue su trayectoria de crecimiento, probablemente debido a las industrias que operan en él.
Observando cada fila y entre filas, es evidente que cada cluster es diferente de los demás.
Algunos parecen crecer continuamente, mientras que otros fluctúan.
Algunos grupos experimentan auges y caídas, posiblemente debido a la influencia de los auges y caídas de las empresas tecnológicas concentradas en ellos.
Otros han experimentado un descenso constante del empleo durante mucho tiempo, probablemente debido a la pérdida de población.
En marcado contraste, otros grupos han experimentado un crecimiento constante a lo largo de todo el conjunto de datos, independientemente de los periodos de contracción.
Unos últimos han experimentado volatilidad en el empleo, la mayoría de los cuales han repuntado algo en los últimos años.

*Utilizar los clusters*.
Desde el punto de vista de las políticas, estas agrupaciones pueden servir de referencia para las intervenciones adaptadas al perfil de cada una de ellas.
Un cluster en declive no debería recibir el mismo tratamiento político que un cluster con buenos recursos o un cluster en continuo crecimiento.
Al mismo tiempo, no necesitamos diseñar 58 intervenciones políticas distintas.
Con la agrupación podemos ir más allá de la "talla única" y responder mejor a las necesidades de los electores.

Desde el punto de vista de la investigación, estas agrupaciones pueden mejorar la calidad de los modelos.
Los economistas y los investigadores financieros suelen estimar modelos econométricos utilizando datos de series temporales de panel.
Una estrategia común emplea una regresión de efectos fijos que permite que cada panel (por ejemplo, condado, estado) tenga su propio intercepto, pero asume que todos los paneles tienen la misma relación con las variables de entrada.
Por ejemplo, imaginemos que se supone que el empleo en el condado de Sierra tiene la misma relación con los insumos que el condado de San Francisco a pesar de las diferencias en las trayectorias de crecimiento.
A su vez, el modelo podría proporcionar estimaciones de coeficientes engañosas, pero también predicciones sesgadas.
HAC puede ayudar a identificar formas estadísticamente defendibles de dividir un panel en grupos más pequeños que muestren un comportamiento similar y mejorar la calidad de la investigación.

# Verificación (10 puntos)

Toda tarea del curso cierra aquí. En agrupamiento la verificación es especialmente necesaria,
porque no hay una métrica de acierto que te avise cuando algo salió mal: el código corre, la
gráfica sale bonita y las etiquetas están mal. Corre los tres chequeos y reporta números.

1. **¿Cuántos condados aparecen realmente en la figura de dispersión? (4 pts)** La celda marcada
   como "no modifiques" grafica `np.log(cty['pct.tech'])`. Cuenta cuántos condados tienen
   `pct.tech == 0` y qué les hace el logaritmo. Reporta: cuántos condados hay en `cty`, cuántos
   puntos son efectivamente dibujables en esa figura, y qué tipo de condado es el que desaparece.
   ¿Cambia eso alguna de las conclusiones del texto que viene después de la figura?

   En el mismo chequeo, verifica la **alineación de las etiquetas**: `km_opt.labels_` tiene un
   color por renglón de `inputs`, y se está usando para colorear renglones de `cty`. Comprueba que
   `len(km_opt.labels_) == len(cty)`. Si en algún punto tiraste renglones con faltantes al
   construir `inputs`, los colores están recorridos y la figura está mintiendo.

2. **Un conteo por dos caminos (3 pts).** Para la sección de series de tiempo: verifica que la
   suma de los tamaños de tus clusters sea exactamente 58 (el número de condados de California) y
   que el número de columnas de `rho` también sea 58 — no 60. Las dos primeras columnas de `cali`
   son de tiempo, no condados; si se colaron, tu matriz de correlaciones tiene dos series que no
   son condados y tus grupos están contaminados. Imprime `rho.shape` y los nombres de las primeras
   y últimas columnas para confirmarlo.

3. **Estabilidad (3 pts).** Vuelve a ajustar `km_opt` con dos semillas distintas
   (`random_state`) y compara los tamaños de los grupos resultantes con los originales. ¿Se
   parecen? Un agrupamiento cuya composición cambia radicalmente al cambiar la semilla no soporta
   las afirmaciones que hiciste sobre él, por más buena que se vea la gráfica.

Cierra con una línea sobre el uso de IA: qué error o decisión cuestionable encontraste en el
trabajo del agente; si no encontraste ninguno, qué revisaste para descartarlos. Si no usaste IA en
esta tarea, decláralo.

In [ ]:
# Chequeo 1: condados con pct.tech == 0 y alineación de km_opt.labels_ con cty.
# Chequeo 2: tamaños de clusters de California y forma de rho.
# Chequeo 3: estabilidad ante cambios de semilla.

**Conclusiones de tu verificación aquí** (edita esta celda de markdown)

1. Condados en `cty`: ___ ; con `pct.tech == 0`: ___ ; puntos dibujables: ___ .
   `len(km_opt.labels_) == len(cty)`: ___ . Qué tipo de condado desaparece:
2. Suma de tamaños de clusters de California: ___ . `rho.shape`: ___ .
3. Tamaños de grupos con las semillas ___ , ___ y ___ :
4. Uso de IA — errores encontrados o qué revisé para descartarlos: